### Import Dependencies

In [ ]:
import openai
from pydantic import BaseModel, Field
import pandas as pd

from qdrant_client import QdrantClient
from qdrant_client import models
import os

from qdrant_client.models import Filter, FieldCondition, MatchValue, VectorParams, Distance, SparseVectorParams, Modifier, PayloadSchemaType, PointStruct, Document, Prefetch, FusionQuery, MatchAny

import numpy as np
import tiktoken

### Qdrant Client for Qdrant Cloud

In [ ]:
qdrant_client = QdrantClient(
    url="https://<CLUSTER_ID>.<REGION>.aws.cloud.qdrant.io",
    api_key=os.getenv("QDRANT_API_KEY")
)

## Items Collection

### Create Qdrant colection for hybrid search

In [ ]:
qdrant_client.create_collection(
    collection_name="Amazon-items-collection-01-hybrid-search",
    vectors_config={
        "text-embedding-3-small": VectorParams(size=1536, distance=Distance.COSINE)
    },
    sparse_vectors_config={
        "bm25": SparseVectorParams(modifier=Modifier.IDF)
    }
)

In [ ]:
qdrant_client.create_payload_index(
    collection_name="Amazon-items-collection-01-hybrid-search",
    field_name="parent_asin",
    field_schema=PayloadSchemaType.KEYWORD
)

### Embedding Functions

In [ ]:
def get_embedding(text, model="text-embedding-3-small"):    
    response = openai.embeddings.create(
        input=text,
        model=model,
    )
    return response.data[0].embedding

In [ ]:
def get_embeddings_batch(text_list, model="text-embedding-3-small", batch_size=100):
    
    if len(text_list) <= batch_size:
        response = openai.embeddings.create(input=text_list, model=model)
        return [embedding.embedding for embedding in response.data]
    
    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i + batch_size]
        response = openai.embeddings.create(input=batch, model=model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        print(f"Processed {counter * batch_size} of {len(text_list)}")
        counter += 1
    
    return all_embeddings

### Read the sampled dataset with Amazon inventory data

In [ ]:
df_items = pd.read_json("../../data/meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines=True)

In [ ]:
df_items.head()

### Preprocess title and features

In [ ]:
def preprocess_description(row):
    return f"{row['title']} {' '.join(row['features'])}"

In [ ]:
def extract_first_large_image(row):
    return row["images"][0].get("large", "")

In [ ]:
df_items["description"] = df_items.apply(preprocess_description, axis=1)
df_items["image"] = df_items.apply(extract_first_large_image, axis=1)

In [ ]:
df_items.head()

In [ ]:
list(df_items["description"].items())[0]

In [ ]:
list(df_items["image"].items())[0]

In [ ]:
data_to_embed = df_items[["description", "image", "rating_number", "price", "average_rating", "parent_asin"]].to_dict(orient="records")

In [ ]:
data_to_embed

In [ ]:
len(data_to_embed)

In [ ]:
text_to_embed = [item["description"] for item in data_to_embed]

In [ ]:
text_to_embed

In [ ]:
embeddings = get_embeddings_batch(text_to_embed)

In [ ]:
len(embeddings)

In [ ]:
pointstructs = []
i = 1
for embedding, data in zip(embeddings, data_to_embed):
    pointstructs.append(
        PointStruct(
            id=i,
            vector={
                "text-embedding-3-small": embedding,
                "bm25": Document(
                    text=data["description"],
                    model="qdrant/bm25"
                )
            },
            payload=data
        )
    )
    i += 1

In [ ]:
pointstructs[0].vector

In [ ]:
qdrant_client.upsert(
    collection_name="Amazon-items-collection-01-hybrid-search",
    points=pointstructs[0:500],
    wait=True
)

In [ ]:
qdrant_client.upsert(
    collection_name="Amazon-items-collection-01-hybrid-search",
    points=pointstructs[500:],
    wait=True
)

### Hybrid Retrieval

In [ ]:
def retrieve_data(query, qdrant_client, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )

    retrieved_context_ids = []
    retieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retieved_context.append(result.payload["description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [ ]:
results = retrieve_data("Can I get a tablet?", qdrant_client, k=20)

In [ ]:
results

### Hybrid Search with weighted RRF

In [ ]:
def retrieve_data(query, qdrant_client, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )
        ],
        query=models.RrfQuery(rrf=models.Rrf(weights=[3,1])),
        limit=k
    )

    retrieved_context_ids = []
    retieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retieved_context.append(result.payload["description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [ ]:
results = retrieve_data("Can I get a tablet?", qdrant_client, k=20)

In [ ]:
results

## Reviews Collection

In [ ]:
qdrant_client.create_collection(
    collection_name="Amazon-reviews-collection-01",
    vectors_config={
        "text-embedding-3-small": VectorParams(size=1536, distance=Distance.COSINE)
    }
)

In [ ]:
qdrant_client.create_payload_index(
    collection_name="Amazon-reviews-collection-01",
    field_name="parent_asin",
    field_schema=PayloadSchemaType.KEYWORD
)

### Embedding Functions

In [ ]:
def get_embedding(text, model="text-embedding-3-small"):    
    response = openai.embeddings.create(
        input=text,
        model=model,
    )
    return response.data[0].embedding

In [ ]:
def get_embeddings_batch(text_list, model="text-embedding-3-small", batch_size=100):
    
    if len(text_list) <= batch_size:
        response = openai.embeddings.create(input=text_list, model=model)
        return [embedding.embedding for embedding in response.data]
    
    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i + batch_size]
        response = openai.embeddings.create(input=batch, model=model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        print(f"Processed {counter * batch_size} of {len(text_list)}")
        counter += 1
    
    return all_embeddings

### Retrieve all item IDs from Amazon Items Qdrant Collection

In [ ]:
dummy_vector = np.zeros(1536).tolist()

In [ ]:
payload = qdrant_client.query_points(
    collection_name="Amazon-items-collection-01-hybrid-search",
    query=dummy_vector,
    using="text-embedding-3-small",
    limit=1000,
    with_payload=["parent_asin"],
    with_vectors=False
)

In [ ]:
payload.points

In [ ]:
len(payload.points)

In [ ]:
parent_asin_list = [item.payload["parent_asin"] for item in payload.points]

In [ ]:
parent_asin_list

### Load Amazon Reviews Dataset

In [ ]:
df_reviews = pd.read_json("../../data/Electornics_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines=True)

In [ ]:
df_reviews.head()

In [ ]:
len(df_reviews)

In [ ]:
df_reviews_sample = df_reviews[df_reviews["parent_asin"].isin(parent_asin_list)]

In [ ]:
len(df_reviews_sample)

### Define functions to preprocess reviews data

In [ ]:
def preprocess_reviews_data(row):
    return f"{row['title']} {row['text']}"

In [ ]:
encoding = tiktoken.encoding_for_model("text-embedding-3-small")

In [ ]:
encoding.encode("Can I get some earphones?")

In [ ]:
def token_count(row, model="text-embedding-3-small"):

    encoding = tiktoken.encoding_for_model(model)

    return len(encoding.encode(row["preprocessed_data"]))

In [ ]:
df_reviews_sample["preprocessed_data"] = df_reviews_sample.apply(preprocess_reviews_data, axis=1)
df_reviews_sample["token_count"] = df_reviews_sample.apply(token_count, axis=1)

In [ ]:
df_reviews_sample.head()

In [ ]:
len(df_reviews_sample)

In [ ]:
df_reviews_sample = df_reviews_sample[df_reviews_sample["token_count"] < 8192]

In [ ]:
len(df_reviews_sample)

In [ ]:
total_tokens = df_reviews_sample["token_count"].sum()

In [ ]:
total_tokens

### Embed the text and add additional filds to the payload of each vector for reviews

In [ ]:
data_to_embed_reviews = df_reviews_sample[["preprocessed_data", "parent_asin"]].to_dict(orient="records")

In [ ]:
data_to_embed_reviews

In [ ]:
len(data_to_embed_reviews)

In [ ]:
text_to_embed_reviews = [item["preprocessed_data"] for item in data_to_embed_reviews]

In [ ]:
text_to_embed_reviews

In [ ]:
embeddings = get_embeddings_batch(text_to_embed_reviews, batch_size=500)

In [ ]:
len(embeddings)

In [ ]:
pointstructs = []
i = 1
for embedding, data in zip(embeddings, data_to_embed_reviews):
    pointstructs.append(
        PointStruct(
            id=i,
            vector={
                "text-embedding-3-small": embedding,
            },
            payload=data
        )
    )
    i += 1

In [ ]:
pointstructs[0].vector

In [ ]:
batch_size_qdrant = 100
counter = 1
for i in range(0, len(pointstructs), batch_size_qdrant):
    batch = pointstructs[i:i + batch_size_qdrant]
    qdrant_client.upsert(
        collection_name="Amazon-reviews-collection-01",
        points=batch,
        wait=True
    )
    print(f"Processed {counter * batch_size_qdrant} of {len(pointstructs)}")
    counter += 1

In [ ]:
import time
import random

from qdrant_client.http.exceptions import ResponseHandlingException, UnexpectedResponse

batch_size_qdrant = 100
max_retries = 5
base_delay = 1.0

for i in range(0, len(pointstructs), batch_size_qdrant):
    batch = pointstructs[i:i + batch_size_qdrant]

    for attempt in range(max_retries):
        try:
            qdrant_client.upsert(
                collection_name="Amazon-reviews-collection-01",
                points=batch,
                wait=True,
            )
            break
        except (ResponseHandlingException, UnexpectedResponse, TimeoutError) as e:
            if attempt == max_retries - 1:
                raise RuntimeError(
                    f"Failed to upsert batch starting at index {i} after {max_retries} attempts"
                ) from e
            delay = base_delay * (2 ** attempt) + random.uniform(0, 1)
            print(f"Batch at {i} failed ({type(e).__name__}: {e}); retrying in {delay:.1f}s")
            time.sleep(delay)

    print(f"Processed {min(i + batch_size_qdrant, len(pointstructs))} of {len(pointstructs)}")